# 04 · Main evaluation (Step B4)

Evaluates one trained run per session on every condition in `configs/eval_ood.yaml`:
- **In-distribution:** LIBERO's fixed test states, 50 rollouts per task.
- **Camera:** yaw ±20° (50), yaw +10°, height +10 cm, field of view +10% (25 each).
- **Language:** the 3 paraphrases per task (25).
- **Position / layout / distractor:** run once `ood/make_ood_states.py` has filled the manifest. Until then they are skipped, never run on the default states.

Every run uses the same initial states and seeds, so the report can compare variants **pairwise on matched episodes**, which is much more sensitive than comparing two separate confidence intervals.

Run order: train each run in `03_train` (it writes `selected_step.txt`), then evaluate each here. Several Colab sessions can evaluate different runs in parallel; results are resumable JSONL on Drive.

In [ ]:
# ---- edit per session ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"
BRANCH = "main"
RUN = "v1_rgb"          # v1_rgb, v1_rgb_s1, v1_aug, v2_mask, v3_grounded, v3_grounded_s1
CONFIGS = {"v1_rgb": "configs/v1_rgb.yaml", "v1_aug": "configs/v1_aug.yaml",
           "v2_mask": "configs/v2_mask.yaml", "v3_grounded": "configs/v3_grounded.yaml"}
CONFIG = CONFIGS[RUN.split("_s")[0]]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

In [ ]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
%cd /content/GroundedVLA
!git pull --ff-only && git log -1 --format="commit %h  %s"
!bash scripts/colab_install.sh requirements.txt

In [ ]:
import json
from pathlib import Path
from groundedvla.utils.colab import setup_session
paths = setup_session()
RUN_DIR = Path(paths["checkpoints"]) / RUN
STEP = int((RUN_DIR / "selected_step.txt").read_text())       # chosen in 03_train, section 3
CKPT = RUN_DIR / f"step_{STEP:06d}"
EVAL_ROOT = Path(paths["outputs"]) / "eval"
print(RUN, "->", CKPT)

## Evaluate (resumable: re-run after a disconnect)

In [ ]:
!python -m groundedvla.eval.run_eval --config {CONFIG} --checkpoint {CKPT} --eval-config configs/eval_ood.yaml --name {RUN} --tag main

## Report: all runs evaluated so far, then paired comparisons

Pooled over seeds per variant. `diff` is A − B on matched episodes with a bootstrap 95% CI: if the interval excludes 0, the difference is real at this sample size.

In [ ]:
mains = sorted(str(p) for p in EVAL_ROOT.glob("*/smolvla_ckpt/main"))
print("\n".join(mains))
out = EVAL_ROOT / "summary_main.csv"
PAIRS = [("v3_grounded", "v1_rgb"),      # total effect of grounding (proposal's comparison)
         ("v3_grounded", "v2_mask"),     # metric 3D beyond object identity (core question)
         ("v2_mask", "v1_rgb"),          # object identity alone
         ("v1_aug", "v1_rgb"),           # does plain augmentation buy robustness?
         ("v3_grounded", "v1_aug")]      # 3D grounding vs. augmentation
!python -m groundedvla.eval.report {" ".join(mains)} --out {out}
for a, b in PAIRS:
    !python -m groundedvla.eval.report {" ".join(mains)} --out {out} --compare {a} {b} | sed -n '/^paired/,$p'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(out)
t = df.pivot_table(index="condition", columns="label", values="success", aggfunc="mean")
display(t.style.format("{:.0%}"))
ax = t.plot.bar(figsize=(12, 4), rot=30, ylim=(0, 1), title="success by condition (all tasks)")
ax.set_ylabel("success"); plt.tight_layout(); plt.show()